# 🚀 07 — Full End-to-End Pipeline Demo

**EvaliSense — AI-Powered Handwritten Examination Evaluation**

This notebook demonstrates the complete evaluation pipeline:

```
Handwritten Image → Preprocessing → Line Segmentation → TrOCR (HTR)
     → Semantic Evaluation → Feature Extraction → Risk Prediction
          → Examiner Review → Ground-Truth Record
```

### Pipeline Architecture
```
┌──────────────┐    ┌──────────────┐    ┌───────────────┐
│  Answer      │───▶│  Preprocess  │───▶│  Line         │
│  Sheet Image │    │  (OpenCV)    │    │  Segmenter    │
└──────────────┘    └──────────────┘    └───────┬───────┘
                                                │
                                                ▼
┌──────────────┐    ┌──────────────┐    ┌───────────────┐
│  Risk        │◀───│  Feature     │◀───│  TrOCR        │
│  Model (SVM) │    │  Extraction  │    │  Recognition  │
└──────┬───────┘    └──────────────┘    └───────┬───────┘
       │                                        │
       ▼                                        ▼
┌──────────────┐    ┌──────────────┐    ┌───────────────┐
│  Examiner    │◀───│  Risk        │◀───│  Rubric-Based │
│  Dashboard   │    │  Alert       │    │  Evaluation   │
└──────────────┘    └──────────────┘    └───────────────┘
```

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, HTML

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)

## Step 1: Load Input Image

In [ ]:
# Use a generated test answer image
image_path = Path('../data/raw/answer_photosynthesis_good.png')

if not image_path.exists():
    # Try another image
    raw_images = list(Path('../data/raw').glob('answer_*.png'))
    if raw_images:
        image_path = raw_images[0]
    else:
        print('No test images found. Run: python create_test_images.py')

original = cv2.imread(str(image_path))
print(f'Input: {image_path.name}')
print(f'Shape: {original.shape}')

fig, ax = plt.subplots(figsize=(10, 14))
ax.imshow(cv2.cvtColor(original[:2000], cv2.COLOR_BGR2RGB))
ax.set_title(f'Input Answer Sheet: {image_path.name}', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

## Step 2: Preprocessing

In [ ]:
from preprocessing import ImagePreprocessor

preprocessor = ImagePreprocessor()

t0 = time.time()
preprocess_result = preprocessor.process(str(image_path))
preprocess_time = time.time() - t0

print(f'✓ Preprocessing complete in {preprocess_time:.2f}s')
print(f'  Input shape:  {preprocess_result.original.shape}')
print(f'  Output shape: {preprocess_result.final.shape}')
print(f'  Metadata: {preprocess_result.metadata}')

## Step 3: HTR — Handwritten Text Recognition

In [ ]:
from htr import HTRPipeline

print('Loading TrOCR model...')
t0 = time.time()
htr_pipeline = HTRPipeline()

htr_result = htr_pipeline.recognize_path(image_path)
htr_time = time.time() - t0

print(f'\n✓ HTR complete in {htr_time:.1f}s')
print(f'  Lines detected: {len(htr_result.lines)}')
print(f'  Average confidence: {htr_result.average_confidence:.4f}' if htr_result.average_confidence else '  Average confidence: N/A')
print(f'\nRecognized text:')
print('─' * 60)
for line in htr_result.lines:
    conf = f'[{line.confidence:.3f}]' if line.confidence else '[N/A]'
    print(f'  Line {line.line_number:2d} {conf}: {line.text}')
print('─' * 60)
print(f'\nFull text ({len(htr_result.full_text)} chars):')
print(htr_result.full_text[:500])

## Step 4: Rubric-Based Answer Evaluation

In [ ]:
from evaluation import Rubric, AnswerEvaluator

# Load the matching rubric
rubric_path = Path('../data/samples/rubric_biology_photosynthesis.json')
with open(rubric_path, 'r', encoding='utf-8') as f:
    rubric = Rubric.from_dict(json.load(f))

print(f'Rubric: {rubric_path.name}')
print(f'Question: {rubric.question[:80]}...')
print(f'Max marks: {rubric.max_marks}')
print(f'Criteria: {len(rubric.criteria)}')

# Evaluate
evaluator = AnswerEvaluator(model_name='all-MiniLM-L6-v2', similarity_threshold=0.55)

t0 = time.time()
eval_result = evaluator.evaluate(htr_result.full_text, rubric)
eval_time = time.time() - t0

print(f'\n✓ Evaluation complete in {eval_time:.2f}s')
print(f'  Total Score: {eval_result.total_score:.1f} / {eval_result.max_score:.0f}')
print(f'  Confidence: {eval_result.overall_confidence:.4f}')

print(f'\n  Criterion-level results:')
for cr in eval_result.criteria_results:
    cr_dict = cr.to_dict()
    print(f'    [{cr_dict["criterion_id"]}] {cr_dict["score"]:.1f}/{cr_dict["max_marks"]:.0f} '
          f'(sim={cr_dict.get("similarity", 0):.3f})')

## Step 5: Feature Extraction

In [ ]:
from features import extract_features, FEATURE_NAMES

feature_vec = extract_features(htr_result.to_dict(), eval_result.to_dict())

print(f'Feature vector ({len(feature_vec.values)} dimensions):')
print(f'{"Feature":<45} {"Value":>10}')
print('─' * 58)
for name, value in zip(feature_vec.names, feature_vec.values):
    bar = '█' * max(0, int(value * 20))
    print(f'{name:<45} {value:>10.4f}  {bar}')

## Step 6: Grading Error Risk Prediction

In [ ]:
from models import GradingRiskModel
from config import config

model_path = config.risk_model_path

if model_path.exists():
    risk_model = GradingRiskModel.load(model_path)
    
    prediction = risk_model.predict(feature_vec.values, feature_vec.names)
    pred_dict = prediction.to_dict()
    
    risk_label = pred_dict['risk_label']
    risk_prob = pred_dict['risk_probability']
    
    color = '\033[91m' if risk_label == 'HIGH' else '\033[92m'
    reset = '\033[0m'
    
    print(f'\n╔══════════════════════════════════════╗')
    print(f'║     GRADING ERROR RISK PREDICTION    ║')
    print(f'╠══════════════════════════════════════╣')
    print(f'║  Risk Label:    {risk_label:<20}   ║')
    print(f'║  Probability:   {risk_prob:<20.4f}   ║')
    print(f'║  Model:         {risk_model.model_name:<20}   ║')
    print(f'╚══════════════════════════════════════╝')
    
    if 'contributing_factors' in pred_dict:
        print(f'\n  Contributing factors:')
        for factor in pred_dict['contributing_factors']:
            print(f'    • {factor}')
else:
    print('No trained model found. Using heuristic risk estimation.')
    print('Train a model: python train_risk_model.py')

## Step 7: Examiner Review Simulation

In [ ]:
# Simulate examiner review
ai_mark = eval_result.total_score
max_marks = eval_result.max_score

# Examiner would see: image, recognized text, AI score, risk alert
# Then assigns their own mark
examiner_mark = ai_mark + np.random.uniform(-1.5, 1.5)  # Simulated
examiner_mark = max(0, min(max_marks, round(examiner_mark, 1)))

disagreement = abs(ai_mark - examiner_mark)

print(f'╔══════════════════════════════════════════╗')
print(f'║          EXAMINER REVIEW PANEL           ║')
print(f'╠══════════════════════════════════════════╣')
print(f'║  AI Score:        {ai_mark:>6.1f} / {max_marks:.0f}            ║')
print(f'║  Examiner Score:  {examiner_mark:>6.1f} / {max_marks:.0f}            ║')
print(f'║  Disagreement:    {disagreement:>6.1f} marks            ║')
print(f'║  Action:          {"MODIFIED" if disagreement > 0.01 else "APPROVED":<20} ║')
print(f'╚══════════════════════════════════════════╝')

if disagreement >= 2.0:
    print(f'\n  ⚠️  GRADING ERROR DETECTED (Δ >= 2.0 marks)')
    print(f'  This record will be added to training data for model improvement.')

## Pipeline Summary

In [ ]:
# Create a summary visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('EvaliSense — End-to-End Pipeline Summary', fontsize=18, fontweight='bold')

# 1. Input image (crop)
ax = axes[0, 0]
ax.imshow(cv2.cvtColor(original[:1000, :], cv2.COLOR_BGR2RGB))
ax.set_title('① Input Image', fontsize=12, fontweight='bold')
ax.axis('off')

# 2. Preprocessed
ax = axes[0, 1]
if preprocess_result.final.ndim == 2:
    ax.imshow(preprocess_result.final[:1000, :], cmap='gray')
else:
    ax.imshow(cv2.cvtColor(preprocess_result.final[:1000, :], cv2.COLOR_BGR2RGB))
ax.set_title('② Preprocessed', fontsize=12, fontweight='bold')
ax.axis('off')

# 3. HTR confidence per line
ax = axes[0, 2]
confs = [l.confidence or 0 for l in htr_result.lines]
line_nums = [l.line_number for l in htr_result.lines]
colors = ['#2ECC71' if c > 0.7 else '#F39C12' if c > 0.5 else '#E74C3C' for c in confs]
ax.barh(line_nums, confs, color=colors, edgecolor='white')
ax.set_xlabel('Confidence')
ax.set_ylabel('Line #')
ax.set_title('③ HTR Confidence', fontsize=12, fontweight='bold')
ax.set_xlim(0, 1)
ax.invert_yaxis()

# 4. Evaluation scores
ax = axes[1, 0]
cr_ids = []
cr_scores = []
cr_max = []
for cr in eval_result.criteria_results:
    d = cr.to_dict()
    cr_ids.append(d['criterion_id'])
    cr_scores.append(d['score'])
    cr_max.append(d['max_marks'])

x = np.arange(len(cr_ids))
ax.bar(x, cr_scores, color='#4A90D9', label='Score', edgecolor='white')
ax.bar(x, [m - s for m, s in zip(cr_max, cr_scores)], bottom=cr_scores,
       color='#E0E0E0', alpha=0.5, label='Remaining', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(cr_ids, fontsize=8, rotation=30)
ax.set_ylabel('Marks')
ax.set_title(f'④ Evaluation: {eval_result.total_score:.1f}/{eval_result.max_score:.0f}',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=8)

# 5. Feature radar
ax = axes[1, 1]
key_features = ['ocr_confidence', 'overall_semantic_similarity',
                'evaluation_confidence', 'rubric_coverage', 'normalized_score']
feat_dict = feature_vec.to_dict()
values = [feat_dict.get(f, 0) for f in key_features]
ax.bar(range(len(key_features)), values, color='#9B59B6', edgecolor='white')
ax.set_xticks(range(len(key_features)))
ax.set_xticklabels([f.replace('_', '\n')[:15] for f in key_features], fontsize=7)
ax.set_ylim(0, 1)
ax.set_ylabel('Value')
ax.set_title('⑤ Key Features', fontsize=12, fontweight='bold')

# 6. Risk prediction gauge
ax = axes[1, 2]
try:
    risk_color = '#E74C3C' if risk_label == 'HIGH' else '#2ECC71'
    ax.pie([risk_prob, 1-risk_prob], colors=[risk_color, '#E0E0E0'],
           startangle=90, counterclock=False)
    ax.text(0, 0, f'{risk_label}\n{risk_prob:.1%}', ha='center', va='center',
            fontsize=16, fontweight='bold', color=risk_color)
    ax.set_title('⑥ Risk Prediction', fontsize=12, fontweight='bold')
except NameError:
    ax.text(0.5, 0.5, 'No model\nloaded', ha='center', va='center',
            fontsize=14, transform=ax.transAxes)
    ax.set_title('⑥ Risk Prediction', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/results/pipeline_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Print final timing summary
total_time = preprocess_time + htr_time + eval_time

print(f'\n{"═" * 55}')
print(f'  PIPELINE EXECUTION SUMMARY')
print(f'{"═" * 55}')
print(f'  Preprocessing:    {preprocess_time:>7.2f}s')
print(f'  HTR Recognition:  {htr_time:>7.2f}s  (includes model loading)')
print(f'  Evaluation:       {eval_time:>7.2f}s')
print(f'  ─────────────────────────')
print(f'  Total:            {total_time:>7.2f}s')
print(f'{"═" * 55}')
print(f'  Device:        CPU (Intel Core Ultra 9 285H)')
print(f'  HTR Model:     microsoft/trocr-base-handwritten')
print(f'  Eval Model:    all-MiniLM-L6-v2')
print(f'  Risk Model:    {"Loaded" if model_path.exists() else "Heuristic"}')
print(f'  Lines found:   {len(htr_result.lines)}')
print(f'  Final score:   {eval_result.total_score:.1f} / {eval_result.max_score:.0f}')
print(f'{"═" * 55}')

## API Integration

The same pipeline is available through the REST API:

```bash
# Start the server
uvicorn api.app:app --host 127.0.0.1 --port 8000 --reload

# Upload an image
curl -X POST http://localhost:8000/api/upload -F "file=@answer_sheet.png"

# Preprocess
curl -X POST http://localhost:8000/api/preprocess/{session_id}

# Run HTR
curl -X POST http://localhost:8000/api/recognize/{session_id}

# Evaluate against a rubric
curl -X POST http://localhost:8000/api/evaluate/{session_id} \
     -F "rubric_json=@rubric.json"

# Predict grading error risk
curl -X POST http://localhost:8000/api/predict-risk/{session_id}

# Submit examiner review
curl -X POST http://localhost:8000/api/submit-review/{session_id} \
     -F "examiner_mark=7.5" -F "examiner_notes=Good answer"
```

## Conclusion

EvaliSense provides a complete, CPU-executable pipeline for:
- **Digitizing** handwritten examination answers (TrOCR)
- **Evaluating** answers against structured rubrics (Sentence-BERT)
- **Predicting** grading errors with ML (SVM/RF/GB)
- **Assisting** examiners — the human remains the final authority

The system is designed as a **decision-support tool**, not a replacement for human judgement.